# Transformers e ensembles — pipeline cache-aware

O notebook pode executar toda a pipeline. Se ela já tiver sido rodada no cluster, as chamadas obrigatoriamente recuperam os caches correspondentes e não repetem o treinamento. A busca usa validação interna; o ranking final usa `val_f1_macro` no `dataset_validation`. Todos os treinos têm limite de 100 épocas, early stopping e restauração do melhor checkpoint.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
here = Path.cwd().resolve()
BACKEND = here if (here / 'machine_learning').exists() else (here / 'backend' if (here / 'backend').exists() else here.parent)
sys.path.insert(0, str(BACKEND))
from machine_learning.cache import ModelCache
from machine_learning.transformer.runner import (
    TRANSFORMER_EMBEDDINGS, run_transformer_finalist, finalize_transformer_pipeline
)
# False: usa o cache e calcula apenas o que estiver faltando.
# True: refaz tudo; para isso, prefira o job SLURM.
FORCE_RETRAIN = False
for embedding in TRANSFORMER_EMBEDDINGS:
    run_transformer_finalist(embedding, force_retrain=FORCE_RETRAIN)
summary = finalize_transformer_pipeline()
print('Protocolo:', summary['protocol_version'], '| campeão:', summary['winner'])

## Busca em três etapas por embedding

In [ ]:
for finalist in summary['finalists']:
    embedding = finalist['embedding']
    search_path = BACKEND / 'experiment_results' / 'transformer' / 'search' / embedding / 'summary.json'
    search = json.loads(search_path.read_text(encoding='utf-8'))
    print(f'\n### {embedding.upper()}')
    for stage in search['stages']:
        rows = [{**candidate['config'],
                 'internal_val_f1_macro': candidate['internal_val_f1_macro'],
                 'internal_val_pr_auc': candidate['internal_val_pr_auc'],
                 'internal_val_recall_scam': candidate['internal_val_recall_scam']}
                for candidate in stage['candidates']]
        frame = pd.DataFrame(rows).sort_values('internal_val_f1_macro', ascending=False)
        print(stage['name'], '— vencedor')
        display(frame.head(10))

## Early stopping dos vencedores (seed visual fixa 42)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, finalist in zip(axes, summary['finalists']):
    seed42 = next(row for row in finalist['seeds'] if row['seed'] == 42)
    history = ModelCache.load_history('transformer', seed42['run_id'])
    ax.plot(history['epoch'], history['train_f1'], label='treino')
    ax.plot(history['epoch'], history['val_f1'], label='validação interna')
    ax.set_title(f"{finalist['embedding']} — {len(history)}/100 épocas")
    ax.set_xlabel('época'); ax.set_ylabel('F1 macro'); ax.legend()
plt.tight_layout(); plt.show()

## Três seeds: teste interno e dataset_validation

In [ ]:
finalists = pd.DataFrame([{
    'embedding': row['embedding'], 'configuração': row['configuration'],
    'test_f1_macro_mean': row['test_f1_macro_mean'], 'test_f1_macro_std': row['test_f1_macro_std'],
    'val_f1_macro_mean': row['val_f1_macro_mean'], 'val_f1_macro_std': row['val_f1_macro_std'],
} for row in summary['finalists']])
display(finalists.style.format({c: '{:.4f}' for c in finalists.columns if 'f1_' in c}).highlight_max(subset=['val_f1_macro_mean']))

## Majority voting, soft voting e soma de logits calibrados

In [ ]:
display(pd.DataFrame(summary['ensembles'])[['name', 'f1_macro', 'pr_auc', 'recall_scam', 'roc_auc']].sort_values('f1_macro', ascending=False))
print('Temperaturas ajustadas somente na validação interna:', summary['temperatures'])

## Diagnósticos finais no dataset_validation

In [ ]:
items = []
for finalist in summary['finalists']:
    seed42 = next(row for row in finalist['seeds'] if row['seed'] == 42)
    items.append((finalist['embedding'], ModelCache.load_prediction_bundle('transformer', seed42['run_id'], 'validation')))
for ensemble in summary['ensembles']:
    items.append((ensemble['name'], ModelCache.load_prediction_bundle('transformer', 'ensemble', ensemble['name'])))
fig, axes = plt.subplots(3, len(items), figsize=(5 * len(items), 12))
for col, (name, bundle) in enumerate(items):
    y, pred, prob = bundle['y_true'], bundle['y_pred'], bundle['y_prob']
    ConfusionMatrixDisplay.from_predictions(y, pred, ax=axes[0, col], colorbar=False)
    RocCurveDisplay.from_predictions(y, prob, ax=axes[1, col])
    PrecisionRecallDisplay.from_predictions(y, prob, ax=axes[2, col])
    axes[0, col].set_title(name)
plt.tight_layout(); plt.show()

## Ranking final por val_f1_macro

In [ ]:
ranking = pd.DataFrame(summary['ranking'])
display(ranking.style.format({'val_f1_macro': '{:.4f}', 'std': '{:.4f}'}).highlight_max(subset=['val_f1_macro']))
print('Vencedor:', summary['winner'])

## Estudos adicionais cache-first

As células abaixo apenas leem artefatos. Se estiverem ausentes, execute `bash backend/slurm/studies/submit_cached_model_studies.sh --resume`.

In [ ]:
import seaborn as sns
STUDY_ROOT = BACKEND / 'experiment_results' / 'transformer' / 'studies'
def read_study_csv(relative):
    path = STUDY_ROOT / relative
    if not path.exists():
        print(f'Artefato ausente: {path}')
        print('Execute: bash backend/slurm/studies/submit_cached_model_studies.sh --resume')
        return pd.DataFrame()
    return pd.read_csv(path)

### Curvas por tamanho: teste interno e validação externa

In [ ]:
size = read_study_csv('dataset_size/aggregate.csv')
if not size.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, metric in zip(axes, ['f1_macro', 'recall_scam', 'pr_auc']):
        for (candidate, split), group in size.groupby(['candidate', 'split']):
            group = group.sort_values('fraction')
            x, y = group['fraction'] * 100, group[f'{metric}_mean']
            ci = group.get(f'{metric}_ci95', pd.Series(0, index=group.index))
            ax.plot(x, y, marker='o', label=f'{candidate} — {split}')
            ax.fill_between(x, y-ci, y+ci, alpha=.12)
        ax.set(title=metric, xlabel='% do treino', ylabel=metric); ax.grid(alpha=.25)
    axes[-1].legend(fontsize=7, bbox_to_anchor=(1.04, 1), loc='upper left')
    plt.tight_layout(); plt.show()

### Complementaridade: quem salva quem

In [ ]:
comp = read_study_csv('complementarity/summary.csv')
if not comp.empty:
    rescue = comp[(comp.analysis == 'pairwise_rescue') & (comp['class'] == 'all')]
    matrix = rescue.groupby(['source', 'rescuer'])['count'].mean().unstack(fill_value=0)
    import seaborn as sns
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.heatmap(matrix, annot=True, fmt='.1f', cmap='Blues', ax=axes[0])
    axes[0].set_title('A errou e B acertou')
    ens = comp[comp.analysis.isin(['ensemble_rescue', 'ensemble_harm'])]
    sns.barplot(data=ens, x='rescuer', y='count', hue='analysis', ax=axes[1])
    axes[1].tick_params(axis='x', rotation=20); axes[1].set_title('Resgates e prejuízos do ensemble')
    plt.tight_layout(); plt.show()

### Threshold histórico: FN=0 na validação interna

In [ ]:
threshold_metrics = read_study_csv('threshold/metrics.csv')
threshold_curve = read_study_csv('threshold/threshold_curve.csv')
if not threshold_metrics.empty:
    view = threshold_metrics[threshold_metrics.split.isin(['test_internal', 'validation'])]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.barplot(data=view, x='variant', y='FN', hue='split', ax=axes[0])
    sns.barplot(data=view, x='variant', y='FP', hue='split', ax=axes[1])
    for ax in axes: ax.tick_params(axis='x', rotation=25)
    plt.tight_layout(); plt.show()
if not threshold_curve.empty:
    for seed, group in threshold_curve.groupby('seed'):
        plt.plot(group.threshold, group.FP, alpha=.7, label=f'FP seed {seed}')
        plt.plot(group.threshold, group.FN, '--', alpha=.7, label=f'FN seed {seed}')
    plt.xlabel('threshold'); plt.ylabel('contagem'); plt.legend(); plt.grid(alpha=.25); plt.show()

### Explicabilidade por layer e head

Atenção é evidência descritiva; ablação e occlusão medem impacto na decisão. Cada posição representa um turno completo.

In [ ]:
heads = read_study_csv('explainability/head_summary.csv')
ablation = read_study_csv('explainability/head_stability.csv')
attention_vs_occlusion = read_study_csv('explainability/attention_occlusion_correlation.csv')
if not heads.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric in zip(axes, ['attention_entropy', 'attended_distance', 'recent_attention']):
        sns.barplot(data=heads, x='head', y=metric, hue='label', errorbar='sd', ax=ax)
        ax.set_title(metric)
    plt.tight_layout(); plt.show()
if not ablation.empty:
    for embedding, group in ablation.groupby('embedding'):
        table = group.groupby(['layer', 'head']).impact_mean.mean().unstack(fill_value=0)
        sns.heatmap(table, annot=True, fmt='.4f', cmap='magma')
        plt.title(f'Impacto causal por ablação — {embedding}'); plt.show()
if not attention_vs_occlusion.empty:
    sns.boxplot(data=attention_vs_occlusion, x='embedding', y='attention_occlusion_correlation', hue='split')
    plt.axhline(0, color='black', lw=1); plt.title('Atenção recebida × impacto por occlusão'); plt.show()

In [ ]:
# Visualizador de um caso representativo sem persistir seu texto nos resultados.
map_files = sorted((STUDY_ROOT / 'explainability').glob('*/seed_42/*/attention_maps/*.npz'))
if map_files:
    selected_map = map_files[0]  # troque o índice para navegar pelos casos
    with np.load(selected_map) as data:
        attention, rollout = data['attention'], data['rollout']
        selected_sample_id = str(data['sample_id'])
        selected_speakers = data['speakers'].astype(str)
    n_layers, n_heads = attention.shape[:2]
    fig, axes = plt.subplots(n_layers, n_heads, figsize=(4*n_heads, 3.5*n_layers), squeeze=False)
    for layer in range(n_layers):
        for head in range(n_heads):
            sns.heatmap(attention[layer, head], cmap='viridis', ax=axes[layer, head], cbar=False)
            axes[layer, head].set_title(f'Layer {layer}, head {head}')
    fig.suptitle(f'{selected_sample_id} — {selected_map.parent.parent.name}'); plt.tight_layout(); plt.show()
    from machine_learning.data import _load_parquet, _parquet_path, _sample_ids_from_df
    import re
    # Layout do mapa: explainability/<embedding>/seed_<n>/<split>/attention_maps/<caso>.npz.
    # O embedding está três níveis acima de seed_<n>; usar parents[2] apontava para seed_42.
    embedding_name = selected_map.parents[3].name
    split_name = selected_map.parents[1].name
    source_split = 'validation' if split_name == 'validation' else 'train'
    # Preserve o corpus completo: o sample_id é derivado antes do split e pode referir uma conversa aumentada.
    source = _load_parquet(
        _parquet_path(source_split, None, embedding_name, transformer=True),
        with_augmented=True,
    )
    source_ids = _sample_ids_from_df(source).astype(str)
    matching_rows = np.flatnonzero(source_ids == selected_sample_id)
    if len(matching_rows) != 1:
        raise KeyError(
            f'Não foi possível resolver exatamente um texto para {selected_sample_id} '
            f'em {embedding_name}/{source_split}: {len(matching_rows)} correspondências.'
        )
    raw_text = source.iloc[matching_rows[0]].text
    chunks = [c.strip() for c in re.split(r'(?=Innocent: |Suspect: )', raw_text) if c.strip()][-100:]
    display(pd.DataFrame({'turn_index': range(len(chunks)), 'speaker': selected_speakers, 'text': chunks}))
else:
    print('Mapas ausentes. Execute: bash backend/slurm/studies/submit_cached_model_studies.sh --only explainability')

### Painel consolidado dos estudos cache-first

Esta seção não executa treino. Ela resume as curvas, o ensemble e os casos salvos pelo job SLURM.

In [ ]:
ensemble_metrics = read_study_csv('complementarity/ensemble_metrics.csv')
cases = read_study_csv('complementarity/cases.csv')
if not size.empty:
    summary_columns = ['candidate', 'fraction', 'split', 'n_samples', 'f1_macro_mean', 'f1_macro_ci95', 'recall_scam_mean', 'precision_scam_mean', 'pr_auc_mean', 'FP_mean', 'FN_mean']
    display(size[summary_columns].sort_values(['candidate', 'split', 'fraction']).style.format({c: '{:.4f}' for c in summary_columns if c.endswith(('_mean', '_ci95'))}))
    full = size[size.fraction.eq(1.0)].pivot(index='candidate', columns='split', values='f1_macro_mean')
    if {'test_internal', 'validation'}.issubset(full.columns):
        full['generalization_gap'] = full['test_internal'] - full['validation']
    display(full.sort_values('validation', ascending=False).style.format('{:.4f}'))
if not ensemble_metrics.empty:
    display(ensemble_metrics.sort_values(['split', 'seed', 'f1_macro'], ascending=[True, True, False]))
if not cases.empty:
    audit_cols = [c for c in ['seed', 'split', 'sample_id', 'label', 'base_prediction', 'ensemble_prediction', 'outcome'] if c in cases]
    display(cases[audit_cols].head(30))

### Perfis dos heads e evidência causal

Atenção mostra onde o modelo olha. Ablação de head e oclusão de turno mostram o impacto da informação na decisão.

In [ ]:
class_comparison = read_study_csv('explainability/head_class_comparison.csv')
occlusion = read_study_csv('explainability/turn_occlusion.csv')
stability = read_study_csv('explainability/head_stability.csv')
if not heads.empty:
    speaker_columns = ['attention_suspect_to_suspect', 'attention_suspect_to_innocent', 'attention_innocent_to_suspect', 'attention_innocent_to_innocent']
    speaker_view = heads.melt(id_vars=['embedding', 'split', 'layer', 'head', 'label'], value_vars=[c for c in speaker_columns if c in heads], var_name='speaker_direction', value_name='attention')
    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    sns.barplot(data=speaker_view, x='speaker_direction', y='attention', hue='label', errorbar='sd', ax=axes[0])
    axes[0].tick_params(axis='x', rotation=30); axes[0].set_title('Atenção por direção de speaker')
    recency = heads.groupby(['embedding', 'layer', 'head'], as_index=False)[['recent_attention', 'attended_distance']].mean()
    sns.scatterplot(data=recency, x='attended_distance', y='recent_attention', hue='embedding', size='head', ax=axes[1])
    axes[1].set_title('Distância atendida versus preferência por turnos recentes')
    plt.tight_layout(); plt.show()
if not class_comparison.empty:
    display(class_comparison.sort_values(['embedding', 'split', 'layer', 'head']).head(100))
if not occlusion.empty:
    top_turns = (occlusion.assign(abs_delta=occlusion.delta_logit.abs())
                 .sort_values('abs_delta', ascending=False)
                 .groupby(['embedding', 'split', 'sample_id'], as_index=False).head(5))
    display(top_turns[['embedding', 'split', 'sample_id', 'turn_index', 'speaker', 'delta_logit']].head(50))
if not stability.empty:
    display(stability.sort_values('impact_mean', ascending=False).head(40))